# Kamera og IP-kamera

`cv2.VideoCapture` virker med kamera her også:

- **Webkamera (UVC):** `cv2.VideoCapture(0)` er standardkameraet, `1` det neste. Nettleseren spør om lov første gang.
- **IP-kamera:** gjennom MediaMTX, som i prosjektet *openCV_webStream*. Se avsnittet nederst.

Kameraene du har åpnet, vises som knapper i verktøylinjen øverst. Et trykk på knappen stopper kameraet, og da gir `cap.read()` `False`. Krever Chrome eller Edge 137 eller nyere.

Kjør cellene én og én med **Shift+Enter**. Løkkene går til du stopper dem, så **Kjør alle** blir stående ved den første.

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)          # webkameraet
if not cap.isOpened():
    print("Fikk ikke åpnet kameraet. Se meldingen over.")
else:
    bredde = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    hoyde = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"{bredde} × {hoyde} piksler, {cap.get(cv2.CAP_PROP_FPS):g} bilder per sekund")
    ok, bilde = cap.read()
    cv2.imshow("et bilde fra kameraet", bilde)

## Direkte, med glidebrytere

Den vanlige OpenCV-løkken: les et bilde, behandle det, vis det, og vent litt med `cv2.waitKey`. Her lages en maske for fargene mellom to grenser i HSV, og glidebryterne fra `cv2.createTrackbar` velger grensene. Mens løkken går, virker glidebryterne med en gang, som i et OpenCV-vindu på PC-en.

Hold en farget ting foran kameraet, og dra i **H min** og **H maks** til bare den synes. Rødt ligger rundt H = 0–10, gult rundt 25–35, grønt rundt 50–80 og blått rundt 100–130.

**Stopp løkken** med **q**: klikk på bildet først, så havner ikke q-en i en celle. Kameraet er da fortsatt på, så du kan kjøre cellene under.

In [ ]:
cv2.namedWindow("maske")
cv2.createTrackbar("H min", "maske", 0, 179, lambda v: None)
cv2.createTrackbar("H maks", "maske", 179, 179, lambda v: None)
cv2.createTrackbar("S min", "maske", 80, 255, lambda v: None)
cv2.createTrackbar("V min", "maske", 80, 255, lambda v: None)

while True:
    ok, bilde = cap.read()
    if not ok:                          # kameraet er stoppet
        break
    hsv = cv2.cvtColor(bilde, cv2.COLOR_BGR2HSV)
    lav = (cv2.getTrackbarPos("H min", "maske"), cv2.getTrackbarPos("S min", "maske"), cv2.getTrackbarPos("V min", "maske"))
    hoy = (cv2.getTrackbarPos("H maks", "maske"), 255, 255)
    maske = cv2.inRange(hsv, lav, hoy)
    cv2.imshow("maske", cv2.bitwise_and(bilde, bilde, mask=maske))
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

print("Grensene:", lav, hoy)

## Ta et bilde, eller ta opp

`cv2.imwrite` lagrer et bilde i Python-mappen. Åpne **Filer** øverst til høyre og trykk **Lagre i bilder/**. Da kan du åpne bildet i Fargelab-fanen og se HSV-verdien til hver piksel med markøren.

`cv2.VideoWriter` tar opp video bilde for bilde. Her blir det 5 sekunder i `opptak.mp4`.

In [ ]:
ok, bilde = cap.read()
if not ok:
    print("Kameraet er stoppet. Kjør den første cellen igjen.")
else:
    cv2.imwrite("kamerabilde.png", bilde)
    print("Lagret kamerabilde.png")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    opptak = cv2.VideoWriter("opptak.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (bredde, hoyde))
    antall = 0
    for i in range(int(5 * fps)):
        ok, bilde = cap.read()
        if not ok:
            break
        opptak.write(bilde)
        antall += 1
    opptak.release()
    print("Lagret opptak.mp4 med", antall, "bilder")

## IP-kamera gjennom MediaMTX

Nettleseren kan ikke lese RTSP fra et IP-kamera selv. MediaMTX henter strømmen fra kameraet og sender den videre til nettleseren over WebRTC (WHEP), akkurat som i *openCV_webStream*:

1. Start MediaMTX med `start.bat` i *openCV_webStream* (kameraets RTSP-adresse står i `mediamtx.yml`, under stien `cam`).
2. Bruk adressen til MediaMTX i `cv2.VideoCapture`: `http://127.0.0.1:8889/cam/whep`, eller `rtsp://127.0.0.1:8554/cam`, som gjøres om til den samme.

Her vises bildet og kantene fra `cv2.Canny` ved siden av hverandre. Stopp med **q** eller knappen i verktøylinjen.

In [ ]:
ip = cv2.VideoCapture("http://127.0.0.1:8889/cam/whep")   # eller "rtsp://127.0.0.1:8554/cam"
if not ip.isOpened():
    print("Fikk ikke åpnet IP-kameraet. Kjører MediaMTX, og heter strømmen cam?")
else:
    print(int(ip.get(cv2.CAP_PROP_FRAME_WIDTH)), "×", int(ip.get(cv2.CAP_PROP_FRAME_HEIGHT)), "piksler")
    while True:
        ok, bilde = ip.read()
        if not ok:
            break
        kanter = cv2.Canny(cv2.cvtColor(bilde, cv2.COLOR_BGR2GRAY), 50, 150)
        cv2.imshow("IP-kamera", np.hstack([bilde, cv2.cvtColor(kanter, cv2.COLOR_GRAY2BGR)]))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    ip.release()

### Verdt å vite

- `cap.release()` stopper kameraet. **Start på nytt** og **Stopp** stopper alle kameraene.
- `cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)` og `cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)` ber webkameraet om en annen oppløsning. Som i OpenCV starter det på 640 × 480.
- Bildene kommer gjennom nettleseren, og fargene og skarpheten kan derfor være litt annerledes enn det OpenCV på PC-en gir fra det samme kameraet.
- `cv2.waitKey` gir tasten du trykket mens løkken gikk, og glidebryterne virker mens den venter. Uten `cv2.waitKey` i løkken kommer tastene ikke fram.